import libraries for posture detection

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
base_options = python.BaseOptions(model_asset_path='pose_landmarker_full.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO, # Chế độ chạy cho Video/Webcam
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

In [ ]:
import numpy as np

def _vector_angle(v1, v2):
    """Tính góc (degrees) giữa hai vector (2D hoặc 3D, cùng chiều)."""
    norm = np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8
    cos_a = np.dot(v1, v2) / norm
    return np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0)))

def _tilt_from_vertical(vec, use_3d=True):
    """Độ lệch nhỏ nhất so với trục dọc (0..90), tránh lỗi ngược chiều trục Y."""
    if use_3d:
        vertical_up = np.array([0.0, -1.0, 0.0])
    else:
        vertical_up = np.array([0.0, -1.0])
    angle = _vector_angle(vec, vertical_up)
    return min(angle, 180.0 - angle)

def cervical_angle(ear, shoulder, use_3d=True):
    """
    Góc cổ: góc giữa vector vai→tai và trục thẳng đứng (Y lên).
    Dùng 3D để phản ánh đầu tới trước (thường có thành phần Z), không chỉ mặt phẳng ảnh.
    """
    if use_3d:
        vec = np.array([ear.x - shoulder.x, ear.y - shoulder.y, ear.z - shoulder.z])
    else:
        vec = np.array([ear.x - shoulder.x, ear.y - shoulder.y])
    return _tilt_from_vertical(vec, use_3d=use_3d)

def hip_angle(shoulder, hip, knee, use_3d=True):
    """
    Góc tại hông giữa vector hông→vai và hông→đầu gối (3D).
    Khi ngồi đúng thường ~80–100°; khi gập mạnh/suôi có thể nhỏ hơn.
    """
    if use_3d:
        v1 = np.array([shoulder.x - hip.x, shoulder.y - hip.y, shoulder.z - hip.z])
        v2 = np.array([knee.x - hip.x, knee.y - hip.y, knee.z - hip.z])
    else:
        v1 = np.array([shoulder.x - hip.x, shoulder.y - hip.y])
        v2 = np.array([knee.x - hip.x, knee.y - hip.y])
    return _vector_angle(v1, v2)

def back_inclination(shoulder, hip, use_3d=True):
    """
    Độ nghiêng lưng: góc giữa vector hông→vai và trục thẳng đứng (Y lên).
    Dùng 3D (mặc định) để gù về phía trước/sau (trục Z) vẫn được tính — phù hợp
    camera nhìn nghiêng hoặc chính diện hơn so với chỉ x,y.
    Khi ngồi làm việc, thân thường nghiêng ~20–35° so với phương thẳng đứng vẫn
    có thể là bình thường; ngưỡng cảnh báo cần theo ngữ cảnh ngồi, không áp dụng
    chuẩn "đứng thẳng ~0°".
    """
    if use_3d:
        vec = np.array([shoulder.x - hip.x, shoulder.y - hip.y, shoulder.z - hip.z])
    else:
        vec = np.array([shoulder.x - hip.x, shoulder.y - hip.y])
    return _tilt_from_vertical(vec, use_3d=use_3d)

def smooth_ema(prev, value, alpha=0.25):
    """Làm mượt giá trị theo khung hình, giảm nhấp nháy đỏ/xanh do nhiễu landmark."""
    if prev is None:
        return value
    return alpha * value + (1.0 - alpha) * prev


def draw_angle_text(frame, text, pos, color):
    """Vẽ text với nền đen để dễ đọc hơn trên mọi nền."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale, thickness = 0.55, 2
    (tw, th), _ = cv2.getTextSize(text, font, scale, thickness)
    x, y = pos
    cv2.rectangle(frame, (x - 2, y - th - 4), (x + tw + 2, y + 4), (0, 0, 0), -1)
    cv2.putText(frame, text, (x, y), font, scale, color, thickness)


def posture_level(cervical, hip, back):
    """Phân loại nhanh mức tư thế để app mobile hiển thị trạng thái."""
    warnings = 0
    if cervical is not None and cervical > CERVICAL_WARN:
        warnings += 1
    if hip is not None and hip < HIP_WARN_LOW:
        warnings += 1
    if back is not None and back > BACK_WARN:
        warnings += 1

    if warnings == 0:
        return "good"
    if warnings == 1:
        return "warning"
    return "bad"


def build_posture_response(timestamp_ms, cervical, hip, back):
    """JSON tối giản: chỉ timestamp + cờ tư thế đúng/sai."""
    any_signal = (cervical is not None) or (hip is not None) or (back is not None)
    is_posture_correct = any_signal and (posture_level(cervical, hip, back) == "good")
    return {
        "timestamp_ms": int(timestamp_ms),
        "is_posture_correct": bool(is_posture_correct),
    }


print("Các hàm tính góc + JSON response đã sẵn sàng.")

In [ ]:
import os
import time
import json
import threading
import requests
from datetime import datetime
from dotenv import load_dotenv
from IPython.display import display, clear_output
import ipywidgets as widgets

load_dotenv(override=True)

# Chỉ số landmark MediaPipe Pose
IDX_EAR      = 7
IDX_SHOULDER = 11
IDX_HIP      = 23
IDX_KNEE     = 25

CERVICAL_WARN  = 32.0
HIP_WARN_LOW   = 55.0
BACK_WARN      = 38.0
VISIBILITY_MIN = 0.45

# --- Supabase upload config ---
SUPABASE_URL = os.getenv("SUPABASE_URL", "").strip()
SUPABASE_ANON_KEY = os.getenv("SUPABASE_ANON_KEY", "").strip()
SUPABASE_TABLE = os.getenv("SUPABASE_TABLE", "posture_events").strip()
ENABLE_SUPABASE_UPLOAD = bool(SUPABASE_URL and SUPABASE_ANON_KEY and SUPABASE_TABLE)

# --- Camera source: đổi URL ở đây ---
STREAM_URL = "http://10.241.10.231:81/stream"

# --- Widget hiển thị log trong notebook ---
LOG_MAX_LINES = 20
log_output = widgets.Output(layout={"border": "1px solid #444", "height": "320px", "overflow_y": "auto"})
log_lines = []

def log_widget(payload):
    global log_lines
    status = "✅ GOOD" if payload["is_posture_correct"] else "❌ BAD"
    line = f"[{datetime.now().strftime('%H:%M:%S')}] {status} | {json.dumps(payload, ensure_ascii=False)}"
    log_lines.append(line)
    if len(log_lines) > LOG_MAX_LINES:
        log_lines = log_lines[-LOG_MAX_LINES:]
    with log_output:
        clear_output(wait=True)
        print("\n".join(log_lines))

display(log_output)

print(f"[camera] dùng stream: {STREAM_URL}")
print(f"[supabase] upload enabled: {ENABLE_SUPABASE_UPLOAD}")

s_cervical = s_hip = s_back = None


def upload_posture_to_supabase(payload):
    if not ENABLE_SUPABASE_UPLOAD:
        return
    url = f"{SUPABASE_URL}/rest/v1/{SUPABASE_TABLE}"
    headers = {
        "apikey": SUPABASE_ANON_KEY,
        "Authorization": f"Bearer {SUPABASE_ANON_KEY}",
        "Content-Type": "application/json",
        "Prefer": "return=minimal",
    }
    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=2.5)
        if resp.status_code >= 300:
            print(f"[supabase] upload failed: {resp.status_code} {resp.text[:180]}")
    except Exception as ex:
        print(f"[supabase] upload error: {ex}")


def clear_supabase():
    if not ENABLE_SUPABASE_UPLOAD:
        return
    try:
        url = f"{SUPABASE_URL}/rest/v1/{SUPABASE_TABLE}?timestamp_ms=gte.0"
        headers = {
            "apikey": SUPABASE_ANON_KEY,
            "Authorization": f"Bearer {SUPABASE_ANON_KEY}",
        }
        resp = requests.delete(url, headers=headers, timeout=5)
        if resp.status_code < 300:
            print("[supabase] đã xóa toàn bộ data.")
        else:
            print(f"[supabase] xóa thất bại: {resp.status_code} {resp.text[:180]}")
    except Exception as ex:
        print(f"[supabase] lỗi khi xóa: {ex}")


# --- Thread đọc frame liên tục, chỉ giữ frame mới nhất ---
class LatestFrameCapture:
    def __init__(self, src):
        self.cap = cv2.VideoCapture(src)
        self.cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        self.frame = None
        self.ok = False
        self._lock = threading.Lock()
        self._stop = False
        t = threading.Thread(target=self._reader, daemon=True)
        t.start()

    def _reader(self):
        while not self._stop:
            ok, frame = self.cap.read()
            with self._lock:
                self.ok = ok
                self.frame = frame

    def read(self):
        with self._lock:
            return self.ok, self.frame.copy() if self.frame is not None else None

    def isOpened(self):
        return self.cap.isOpened()

    def get(self, prop):
        return self.cap.get(prop)

    def release(self):
        self._stop = True
        self.cap.release()


with vision.PoseLandmarker.create_from_options(options) as landmarker:
    cap = LatestFrameCapture(STREAM_URL)
    _ts_offset = int(time.time() * 1000)

    while cap.isOpened():
        success, frame = cap.read()
        if not success or frame is None:
            time.sleep(0.01)
            continue

        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)

        timestamp_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))
        if timestamp_ms == 0:
            timestamp_ms = int(time.time() * 1000) - _ts_offset

        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.pose_landmarks:
            pose_lm  = result.pose_landmarks[0]
            world_lm = result.pose_world_landmarks[0]

            h, w = frame.shape[:2]

            for lm in pose_lm:
                cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 3, (0, 220, 0), -1)

            ear      = world_lm[IDX_EAR]
            shoulder = world_lm[IDX_SHOULDER]
            hip      = world_lm[IDX_HIP]
            knee     = world_lm[IDX_KNEE]
            lm_ear      = pose_lm[IDX_EAR]
            lm_shoulder = pose_lm[IDX_SHOULDER]
            lm_hip      = pose_lm[IDX_HIP]
            lm_knee     = pose_lm[IDX_KNEE]

            def _vis(lm):
                return getattr(lm, "visibility", 1.0)

            vis_ok = (
                _vis(lm_ear)          >= VISIBILITY_MIN
                and _vis(lm_shoulder) >= VISIBILITY_MIN
                and _vis(lm_hip)      >= VISIBILITY_MIN
                and _vis(lm_knee)     >= VISIBILITY_MIN
            )

            if vis_ok:
                ang_cervical = cervical_angle(ear, shoulder)
                ang_hip      = hip_angle(shoulder, hip, knee)
                ang_back     = back_inclination(shoulder, hip)
                s_cervical   = smooth_ema(s_cervical, ang_cervical)
                s_hip        = smooth_ema(s_hip, ang_hip)
                s_back       = smooth_ema(s_back, ang_back)
            else:
                ang_cervical = ang_hip = ang_back = None

            p_ear      = (int(lm_ear.x * w),      int(lm_ear.y * h))
            p_shoulder = (int(lm_shoulder.x * w), int(lm_shoulder.y * h))
            p_hip      = (int(lm_hip.x * w),      int(lm_hip.y * h))
            p_knee     = (int(lm_knee.x * w),      int(lm_knee.y * h))

            if vis_ok and s_cervical is not None:
                color_cervical = (0, 0, 255) if s_cervical > CERVICAL_WARN else (0, 255, 150)
                color_hip      = (0, 0, 255) if s_hip < HIP_WARN_LOW       else (0, 255, 255)
                color_back     = (0, 0, 255) if s_back > BACK_WARN         else (100, 200, 255)
            else:
                color_cervical = color_hip = color_back = (0, 255, 0)

            cv2.line(frame, p_ear, p_shoulder, color_cervical, 2)
            cv2.line(frame, p_shoulder, p_hip, color_back, 2)
            cv2.line(frame, p_hip, p_knee, color_hip, 2)

            if vis_ok:
                draw_angle_text(frame, f"Goc co (Cervical): {ang_cervical:.1f}",  (p_ear[0] + 10, p_ear[1]),                color_cervical)
                draw_angle_text(frame, f"Goc hong (Hip): {ang_hip:.1f}",           (p_hip[0] + 10, p_hip[1]),                color_hip)
                draw_angle_text(frame, f"Nghieng lung (Back): {ang_back:.1f}",     (p_shoulder[0] + 10, p_shoulder[1] + 28), color_back)
            else:
                draw_angle_text(frame, "Diem mo / visibility thap", (10, 30), (140, 140, 140))

            response = build_posture_response(timestamp_ms=timestamp_ms, cervical=s_cervical, hip=s_hip, back=s_back)
        else:
            response = build_posture_response(timestamp_ms=timestamp_ms, cervical=None, hip=None, back=None)

        log_widget(response)
        upload_posture_to_supabase(response)

        cv2.imshow('Posture Analysis', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    clear_supabase()


In [ ]:
import os, requests
from dotenv import load_dotenv
load_dotenv(override=True)

SUPABASE_URL      = os.getenv("SUPABASE_URL", "").strip()
SUPABASE_ANON_KEY = os.getenv("SUPABASE_ANON_KEY", "").strip()
SUPABASE_TABLE    = os.getenv("SUPABASE_TABLE", "posture_events").strip()

resp = requests.delete(
    f"{SUPABASE_URL}/rest/v1/{SUPABASE_TABLE}?timestamp_ms=gte.0",
    headers={
        "apikey": SUPABASE_ANON_KEY,
        "Authorization": f"Bearer {SUPABASE_ANON_KEY}",
    },
    timeout=5,
)
if resp.status_code < 300:
    print(f"✅ Đã xóa toàn bộ data trong bảng '{SUPABASE_TABLE}'.")
else:
    print(f"❌ Xóa thất bại: {resp.status_code} {resp.text[:200]}")
